In [20]:
import time
from typing import Dict, List, Optional
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# Import Colab kernel output bridge for reliable browser updates
try:
    from google.colab import output
    COLAB_AVAILABLE = True
except ImportError:
    COLAB_AVAILABLE = False

# Log In

class Account:
    def __init__(self, username: str, pin: str, tier_name: str, default_crates: int):
        self.username = username
        self._pin = pin
        self.tier_name = tier_name
        self.default_crates = default_crates
        self.order_history: List[dict] = []
        self.active_production_jobs: List[dict] = []

    def verify_pin(self, input_pin: str) -> bool:
        return self._pin == input_pin

    def add_production_job(self, crates_needed: int, duration_seconds: float) -> None:
        job = {
            "crates": crates_needed,
            "start_time": time.time(),
            "completion_time": time.time() + duration_seconds,
            "duration": duration_seconds,
            "status": "IN_PROGRESS"
        }
        self.active_production_jobs.append(job)

    def get_remaining_timers(self) -> List[dict]:
        now = time.time()
        active_list = []
        for job in self.active_production_jobs:
            remaining = max(0.0, job["completion_time"] - now)
            if remaining == 0:
                job["status"] = "COMPLETED"
            active_list.append({
                "crates": job["crates"],
                "remaining_seconds": remaining,
                "status": job["status"]
            })
        return active_list

# Operations and Variables

class CrateStorage:
    def __init__(self, initial_crates: int = 5, minutes_per_passive_crate: float = 5.0, time_scale_factor: float = 1.0):
        self._current_crates = initial_crates
        self.seconds_per_crate = (minutes_per_passive_crate * 60) / time_scale_factor
        self.last_passive_update = time.time()

    def update_passive_generation(self) -> int:
        now = time.time()
        elapsed_seconds = now - self.last_passive_update
        crates_to_add = int(elapsed_seconds // self.seconds_per_crate)
        if crates_to_add > 0:
            self._current_crates += crates_to_add
            self.last_passive_update += crates_to_add * self.seconds_per_crate
        return self._current_crates

    def get_stock(self) -> int:
        self.update_passive_generation()
        return self._current_crates

    def deduct_stock(self, crates: int) -> bool:
        self.update_passive_generation()
        if self._current_crates >= crates:
            self._current_crates -= crates
            return True
        return False


class DemandAnalyzer:
    def __init__(self):
        self.global_order_log: List[int] = []

    def log_order(self, crate_quantity: int) -> None:
        self.global_order_log.append(crate_quantity)

    def forecast_next_demand(self) -> int:
        if not self.global_order_log:
            return 3
        recent = self.global_order_log[-5:]
        return max(1, int(sum(recent) / len(recent)))

# Backend

class SmartFactoryEngine:
    def __init__(self, storage: CrateStorage, analyzer: DemandAnalyzer, time_scale_factor: float = 1.0):
        self.storage = storage
        self.analyzer = analyzer
        self.time_scale_factor = time_scale_factor

    def process_order(self, account: Account, requested_crates: int) -> dict:
        self.storage.update_passive_generation()
        available = self.storage.get_stock()

        if available >= requested_crates:
            self.storage.deduct_stock(requested_crates)
            self.analyzer.log_order(requested_crates)
            account.order_history.append({"crates": requested_crates, "status": "FULFILLED"})
            return {
                "status": "FULFILLED",
                "message": f"⚡ Order of {requested_crates} crate(s) fulfilled IMMEDIATELY from available stock! Stock left: {self.storage.get_stock()} crates."
            }
        else:
            missing_crates = requested_crates - available
            production_time_minutes = missing_crates * 5.0
            production_time_seconds = (production_time_minutes * 60) / self.time_scale_factor

            if available > 0:
                self.storage.deduct_stock(available)

            account.add_production_job(missing_crates, production_time_seconds)
            self.analyzer.log_order(requested_crates)

            return {
                "status": "QUEUED_FOR_PRODUCTION",
                "message": f"🏭 Deficit detected! Used {available} available stock. Factory queued {missing_crates} crate(s). Production time: {production_time_minutes:.1f} mins ({production_time_seconds:.1f}s)."
            }


# User Interface


class SmartProductionUI:

    def __init__(self):
        self.time_scale = 1.0
        self.storage = CrateStorage(initial_crates=5, time_scale_factor=self.time_scale)
        self.analyzer = DemandAnalyzer()
        self.factory = SmartFactoryEngine(self.storage, self.analyzer, time_scale_factor=self.time_scale)

        self.users: Dict[str, Account] = {
            "small_retailer": Account("small_retailer", "1111", tier_name="Low Volume", default_crates=1),
            "medium_market": Account("medium_market", "2222", tier_name="Medium Volume", default_crates=4),
            "bulk_distributor": Account("bulk_distributor", "3333", tier_name="Bulk Volume", default_crates=10)
        }
        self.current_user: Optional[Account] = None

        if COLAB_AVAILABLE:
            output.register_callback('refresh_colab_dashboard', self._colab_callback_refresh)

        self._build_login_screen()
        self._build_dashboard_screen()

        self.root_container = widgets.VBox([self.login_layout])

    def _colab_callback_refresh(self):
        if self.current_user:
            self._update_dashboard()

    def _build_login_screen(self):
        self.login_header = widgets.HTML(
            value="<h2 style='color: #4CAF50; margin-bottom:4px;'>🌱 SDG 12 Smart Production System</h2>"
                  "<h4 style='color: #94a3b8; margin-top:0px;'>🔐 Secure Login Required</h4>"
                  "<hr style='border:1px solid #4CAF50; margin-bottom:15px;'>"
        )

        self.login_user_dropdown = widgets.Dropdown(
            options=[
                ("🛒 small_retailer (Low Volume)", "small_retailer"),
                ("🏪 medium_market (Medium Volume)", "medium_market"),
                ("🏬 bulk_distributor (Bulk Volume)", "bulk_distributor")
            ],
            value="small_retailer",
            description="Select Account:",
            style={'description_width': 'initial'}
        )

        self.login_pin_input = widgets.Password(
            value="",
            placeholder="Enter PIN (e.g., 1111)",
            description="PIN Code:",
            style={'description_width': 'initial'}
        )

        self.btn_login = widgets.Button(
            description="Login",
            button_style="primary",
            icon="key"
        )
        self.btn_login.on_click(self._handle_login)

        self.login_msg = widgets.HTML(value="")

        login_form = widgets.VBox([
            self.login_user_dropdown,
            self.login_pin_input,
            self.btn_login,
            self.login_msg
        ], layout=widgets.Layout(padding="15px", border="1px solid #334155", border_radius="8px", background_color="#1e293b", width="420px"))

        self.login_layout = widgets.VBox([self.login_header, login_form])

    def _handle_login(self, b):
        selected_username = self.login_user_dropdown.value
        entered_pin = self.login_pin_input.value.strip()

        account = self.users.get(selected_username)
        if account and account.verify_pin(entered_pin):
            self.current_user = account
            self.login_pin_input.value = ""
            self.login_msg.value = ""
            self._update_dashboard()
            self.root_container.children = [self.dashboard_layout]
        else:
            self.login_msg.value = "<p style='color:#ef4444; font-weight:bold; margin-top:8px;'>❌ Incorrect PIN! Access Denied.</p>"

    def _handle_logout(self, b):
        self.current_user = None
        with self.console_output:
            clear_output()
        self.root_container.children = [self.login_layout]

    def _handle_clear_completed(self, b):
        if self.current_user:
            now = time.time()
            # Direct 1-line cleanup in UI handler
            self.current_user.active_production_jobs = [
                j for j in self.current_user.active_production_jobs if j["completion_time"] > now
            ]
            self._update_dashboard()
            with self.console_output:
                clear_output()
                print("🧹 Cleared finished jobs from timer list.")

    def _build_dashboard_screen(self):
        self.dashboard_header = widgets.HTML(
            value="<h2 style='color: #4CAF50; margin-bottom:4px;'>🌱 SDG 12 Smart Production System</h2><hr style='border:1px solid #4CAF50; margin-bottom:12px;'>"
        )

        self.btn_logout = widgets.Button(
            description="Logout Account",
            button_style="danger",
            icon="sign-out-alt",
            layout=widgets.Layout(width="140px")
        )
        self.btn_logout.on_click(self._handle_logout)

        self.btn_clear_completed = widgets.Button(
            description="Clear Finished Jobs",
            button_style="warning",
            icon="broom",
            layout=widgets.Layout(width="160px")
        )
        self.btn_clear_completed.on_click(self._handle_clear_completed)

        self.metrics_html = widgets.HTML()

        self.order_dropdown = widgets.Dropdown(
            options=[
                ("⚡ Default Volume Tier", "default"),
                ("📦 Low Volume (1 Crate)", 1),
                ("📦📦 Medium Volume (4 Crates)", 4),
                ("📦📦📦 Bulk Volume (10 Crates)", 10),
                ("Custom Amount ->", "custom")
            ],
            value="default",
            description="Order Size:",
            style={'description_width': 'initial'}
        )

        self.custom_int_input = widgets.IntText(
            value=1,
            description="Custom Qty:",
            disabled=True,
            layout=widgets.Layout(width='180px'),
            style={'description_width': 'initial'}
        )
        self.order_dropdown.observe(self._on_dropdown_change, names='value')

        self.btn_place_order = widgets.Button(
            description="Place Order",
            button_style="success",
            icon="shopping-cart"
        )
        self.btn_place_order.on_click(self._on_place_order)

        self.console_output = widgets.Output(
            layout=widgets.Layout(border="1px solid #475569", padding="10px", min_height="90px")
        )

        order_box = widgets.HBox([self.order_dropdown, self.custom_int_input, self.btn_place_order])
        action_buttons = widgets.HBox([self.btn_clear_completed, self.btn_logout], layout=widgets.Layout(gap="10px"))
        top_bar = widgets.HBox([self.dashboard_header], layout=widgets.Layout(justify_content="space-between", align_items="center"))

        self.dashboard_layout = widgets.VBox([
            top_bar,
            widgets.HBox([action_buttons], layout=widgets.Layout(justify_content="flex-end")),
            self.metrics_html,
            widgets.HTML("<b style='font-size:14px;'>🛒 Place New Order:</b>"),
            order_box,
            widgets.HTML("<br><b style='font-size:14px;'>📋 Activity & Production Console:</b>"),
            self.console_output
        ])

    def _on_dropdown_change(self, change):
        if change['new'] == "custom":
            self.custom_int_input.disabled = False
        else:
            self.custom_int_input.disabled = True

    def _update_dashboard(self):
        if not self.current_user:
            return

        user = self.current_user
        stock = self.storage.get_stock()
        forecast = self.analyzer.forecast_next_demand()
        timers = user.get_remaining_timers()

        if timers:
            timer_items = []
            for j in timers:
                if j["remaining_seconds"] > 0:
                    timer_items.append(
                        f"<li style='color:#f8fafc; margin:4px 0;'>🏭 Job: <b style='color:#38bdf8;'>{j['crates']} Crates</b> | ⏳ <b style='color:#fbbf24;'>{j['remaining_seconds']/60:.1f} mins ({j['remaining_seconds']:.0f}s) remaining</b></li>"
                    )
                else:
                    timer_items.append(
                        f"<li style='color:#f8fafc; margin:4px 0;'>🏭 Job: <b style='color:#38bdf8;'>{j['crates']} Crates</b> | <b style='color:#4ade80; background:#166534; padding:2px 8px; border-radius:4px;'>✅ COMPLETED</b></li>"
                    )
            timer_str = f"<ul style='margin:4px 0; padding-left:20px;'>{''.join(timer_items)}</ul>"
        else:
            timer_str = "<i style='color:#94a3b8; font-size:13px;'>No active factory production jobs running.</i>"

        html_content = f"""
        <div style="background-color: #1e293b; color: #f8fafc; padding: 16px; border-radius: 8px; margin-top: 10px; margin-bottom: 15px; border-left: 6px solid #10b981; box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.3);">
            <p style="margin:4px 0; color: #f8fafc; font-size:14px;">
                <b style="color:#94a3b8;">User Profile:</b> <span style="color:#ffffff; font-weight:bold;">{user.username}</span>
                <span style="background:#334155; color:#60a5fa; padding:3px 10px; border-radius:12px; font-weight:bold; font-size:12px; margin-left:8px;">{user.tier_name}</span>
            </p>
            <p style="margin:4px 0; color: #cbd5e1; font-size:14px;">
                <b style="color:#94a3b8;">Default Order Preference:</b> <span style="color:#ffffff;">{user.default_crates} Crates</span>
            </p>
            <hr style="border:0; border-top: 1px solid #334155; margin: 10px 0;">
            <p style="margin:4px 0; color: #f8fafc; font-size:15px;">
                📦 <b style="color:#94a3b8;">Live Inventory Stock:</b>
                <span style="font-size:20px; color:#4ade80; font-weight:bold; margin: 0 6px;">{stock} Crates</span>
                <i style="color:#94a3b8; font-size:12px;">(Adds +1 crate every 5 mins)</i>
            </p>
            <p style="margin:4px 0; color: #f8fafc; font-size:14px;">
                📈 <b style="color:#94a3b8;">Predicted Demand Next Cycle:</b>
                <b style="color:#facc15; font-size:16px;">{forecast} Crates</b>
            </p>
            <hr style="border:0; border-top: 1px solid #334155; margin: 10px 0;">
            <p style="margin:4px 0; color: #94a3b8; font-weight:bold; font-size:12px; letter-spacing: 0.5px;">ACCOUNT FACTORY TIMERS:</p>
            {timer_str}
        </div>
        """
        self.metrics_html.value = html_content

    def _on_place_order(self, b):
        sel = self.order_dropdown.value
        if sel == "default":
            qty = self.current_user.default_crates
        elif sel == "custom":
            qty = self.custom_int_input.value
        else:
            qty = int(sel)

        forecast = self.analyzer.forecast_next_demand()

        if qty > forecast:
            self._show_confirmation_dialog(qty, forecast)
        else:
            self._execute_order(qty)

    def _show_confirmation_dialog(self, qty: int, forecast: int):
        with self.console_output:
            clear_output()
            print(f"⚠️ [SDG 12 OVERPRODUCTION WARNING]")
            print(f"Order quantity ({qty} crates) exceeds predicted demand ({forecast} crates).")
            print("Producing excess inventory risks material waste and unnecessary energy usage.")
            print("Please confirm if you wish to proceed or cancel:\n")

        btn_proceed = widgets.Button(description="Continue Order", button_style="warning", icon="check")
        btn_cancel = widgets.Button(description="Cancel Order", button_style="danger", icon="times")
        confirm_box = widgets.HBox([btn_proceed, btn_cancel])

        def on_proceed(b):
            confirm_box.close()
            self._execute_order(qty)

        def on_cancel(b):
            confirm_box.close()
            with self.console_output:
                clear_output()
                print("❌ Order CANCELLED by user to prevent potential overproduction waste.")

        btn_proceed.on_click(on_proceed)
        btn_cancel.on_click(on_cancel)

        with self.console_output:
            display(confirm_box)

    def _execute_order(self, qty: int):
        res = self.factory.process_order(self.current_user, qty)
        self._update_dashboard()
        with self.console_output:
            clear_output()
            print(res["message"])

    def display_ui(self):
        display(self.root_container)

        if COLAB_AVAILABLE:
            display(HTML("""
            <script>
            if (window.colabTicker) clearInterval(window.colabTicker);
            window.colabTicker = setInterval(function() {
                if (typeof google !== 'undefined' && google.colab && google.colab.kernel) {
                    google.colab.kernel.invokeFunction('refresh_colab_dashboard', [], {});
                }
            }, 1000);
            </script>
            """))

# =====================================================================
# EXECUTE UI IN COLAB
# =====================================================================
app = SmartProductionUI()
app.display_ui()